# T-KGAT Stage 2 — đối chứng hoán vị ngày fact (`tkgat_kg_perm`)

Notebook này là bản sao của notebook 08 (Phôn), chỉ đổi phần chạy: thay vì quét lưới γ của `tkgat_kg`,
nó chạy **đối chứng hoán vị ngày fact** `configs/model/tkgat_kg_perm.yaml`.
Đối chứng lấy đúng điểm (γ, lr, l2, l2_kg, mess_dropout) mà `tkgat_kg` đã chọn trên validation,
rồi chạy lại 5 seed 2019–2023 với ngày của các fact bị xáo trộn (`permute_kg_ts`).

**Điều kiện:** Recovery phải là ZIP cuối của Stage 2, trong đó `tkgat_kg` đã **hoàn tất**
(có `tkgat_kg_summary.json` và đủ 5 seed test). Nếu chưa, notebook dừng ngay với thông báo rõ ràng.

Bản này chạy **2 lượt** (seed 2022 và 2023): 3 seed đầu (2019–2021) đã xong và nằm trong Recovery. ZIP được lưu sau mỗi lượt.
Chọn **Save Version → Save & Run All**, bật GPU T4 và kiểm tra log
`Running tkgat_kg_perm only` trước khi đóng tab/tắt máy.


In [ ]:
from pathlib import Path

PAYLOAD_DIR = None
# Select a ZIP or results directory if automatic discovery finds several recoveries.
RESTORE_SOURCE = None
PROJECT_ROOT = Path('/kaggle/working/t-kgat-stage2')
RUN_TRAINING = True
NEW_PERM_FITS_THIS_SESSION = 2
CHECKPOINT_ZIP = Path('/kaggle/working/tkgat-lastfm-stage2-perm-results.zip')


## 1. Nạp code/data và khôi phục kết quả Stage 2

Dùng ZIP `tkgat-lastfm-stage2-results.zip` cuối cùng (Stage 2 đã hoàn tất, gamma đã chọn)
làm Recovery duy nhất trong Input, cùng Dataset code/data ban đầu.
Gắn một bản Recovery, hoặc đặt `RESTORE_SOURCE` nếu có nhiều bản trong Input.


In [ ]:
import hashlib
import json
import os
import runpy
import subprocess
import sys
from pathlib import Path, PurePosixPath
from zipfile import ZIP_DEFLATED, ZipFile, is_zipfile


def recovery_archive(input_root, working, expected_identity, explicit=None):
    """Resolve one matching ZIP or extracted results directory, never guess between copies."""
    if explicit is not None:
        location = Path(explicit)
        if not location.exists():
            raise RuntimeError(f'Recovery path does not exist: {location}')
        locations = [location]
    else:
        locations = sorted(input_root.rglob('*.zip'))
        locations += sorted(p.parent for p in input_root.rglob('kaggle-run.json')
                            if p.parent.name == 'results')
    matches = []
    inspected = []
    for location in locations:
        if location.is_file() and is_zipfile(location):
            with ZipFile(location) as archive:
                marker = 'results/kaggle-run.json'
                if marker not in archive.namelist():
                    continue
                saved_identity = json.loads(archive.read(marker))
            candidate = ('zip', location)
        elif location.is_dir():
            folder = location if location.name == 'results' else location / 'results'
            marker = folder / 'kaggle-run.json'
            if not marker.is_file():
                continue
            saved_identity = json.loads(marker.read_text(encoding='utf-8'))
            candidate = ('directory', folder)
        else:
            continue
        inspected.append(str(location))
        if saved_identity == expected_identity:
            matches.append(candidate)
    if not matches:
        raise RuntimeError(
            'No matching Recovery found. Add the previous results as Input; '
            'both ZIP and extracted results/ are supported. '
            f'Found result locations: {inspected}. Refusing to start Stage 2 without KGAT Recovery.'
        )
    if len(matches) != 1:
        raise RuntimeError(f'Multiple recoveries match: {matches}. Set RESTORE_SOURCE explicitly.')
    kind, source = matches[0]
    print('Recovery selected:', source, flush=True)
    if kind == 'zip':
        return source
    target = working / 'kgat-recovery-normalized.zip'
    with ZipFile(target, 'w', compression=ZIP_DEFLATED) as archive:
        for folder in (source, source.parent / 'reports'):
            if folder.is_dir():
                for path in sorted(folder.rglob('*')):
                    if path.is_file():
                        archive.write(path, path.relative_to(source.parent).as_posix())
    return target


def require_perm_inputs(suite):
    """Validate that KGAT and Stage 2 (tkgat_kg) are complete and the control config is the expected one."""
    if not suite['complete']('kgat'):
        raise RuntimeError('KGAT is incomplete in Recovery. Use the completed Stage 2 ZIP.')
    print('kgat: complete (verified)', flush=True)
    if not suite['complete']('tkgat_kg'):
        raise RuntimeError(
            'tkgat_kg is not complete in Recovery (no tkgat_kg_summary.json or fewer than 5 test seeds). '
            'The permutation control needs the gamma that Stage 2 selected. '
            'Finish Stage 2 with notebook 08 and use its final ZIP as Recovery.'
        )
    print('tkgat_kg: complete (verified)', flush=True)
    import yaml

    model_path = suite['ROOT'] / 'configs/model/tkgat_kg_perm.yaml'
    config = yaml.safe_load(model_path.read_text(encoding='utf-8'))
    if (config.get('name') != 'tkgat' or config.get('time_on') != 'kg'
            or config.get('control') != 'permute_kg_ts' or config.get('from_tkgat') != 'tkgat_kg'):
        raise RuntimeError('Expected the Stage 2 permutation control (permute_kg_ts from tkgat_kg).')
    if config.get('seeds') != list(range(2019, 2024)):
        raise RuntimeError('The control requires all five seeds 2019-2023.')
    if config.get('grid'):
        raise RuntimeError('The control takes its point from tkgat_kg; remove the grid.')
    runner = runpy.run_path(str(suite['ROOT'] / 'scripts/run.py'))
    folder = suite['ROOT'] / 'results/runs' / suite['RUN_NAME']
    points, base = runner['tkgat_points'](config, model_path, folder)
    print('Point selected by tkgat_kg:', json.dumps(base, sort_keys=True), flush=True)
    print('Control: fact dates permuted, seeds=[2019, 2020, 2021, 2022, 2023]; 5 fits.', flush=True)
    return points


input_root = Path('/kaggle/input')
working = Path('/kaggle/working')
if PAYLOAD_DIR is None:
    manifests = sorted(input_root.rglob('payload-manifest.json'))
    if not manifests:
        packages = sorted(input_root.rglob('tkgat-lastfm-latest-kaggle.zip'))
        if len(packages) != 1:
            raise RuntimeError('Add the original code/data Dataset as Input, or set PAYLOAD_DIR.')
        unpacked = working / 'tkgat-payload'
        with ZipFile(packages[0]) as archive:
            for name in archive.namelist():
                path = PurePosixPath(name)
                if path.is_absolute() or '..' in path.parts or '\\' in name:
                    raise ValueError(f'Invalid payload member: {name}')
            archive.extractall(unpacked)
        manifests = [unpacked / 'payload-manifest.json']
    if len({path.read_text(encoding='utf-8') for path in manifests}) != 1:
        raise RuntimeError('Multiple code/data payloads found. Set PAYLOAD_DIR.')
    payload_root = manifests[0].parent
else:
    payload_root = Path(PAYLOAD_DIR)
manifest = json.loads((payload_root / 'payload-manifest.json').read_text(encoding='utf-8'))
bundle = payload_root / 'source.bundle'
if hashlib.sha256(bundle.read_bytes()).hexdigest() != manifest['files_sha256']['source.bundle']:
    raise ValueError('Source bundle hash mismatch')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', str(bundle), str(PROJECT_ROOT)], check=True)
commit = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
if commit != manifest['source_commit']:
    raise RuntimeError('Working copy belongs to another payload. Start a fresh session.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
suite = runpy.run_path(str(PROJECT_ROOT / 'scripts/kaggle_lastfm.py'))
suite['prepare'](payload_root)
has_progress = (PROJECT_ROOT / 'results/registry.csv').exists() or any(
    (PROJECT_ROOT / 'results/runs').rglob('*.pt')
)
if not has_progress:
    restore = recovery_archive(input_root, working, suite['identity'](), RESTORE_SOURCE)
    suite['extract_results'](restore)
else:
    print('Keeping existing working progress; Recovery will not overwrite it.', flush=True)
suite['record_identity']()
require_perm_inputs(suite)
suite['backup'](CHECKPOINT_ZIP)
print('Source snapshot:', commit, flush=True)
print('Recovery verified. The Stage 2 permutation control is the only model enabled.', flush=True)

## 2. Chạy đối chứng hoán vị ngày fact

Log phải có `kgat: complete (verified)`, `tkgat_kg: complete (verified)`, dòng
`Point selected by tkgat_kg` và `Running tkgat_kg_perm only`.
Mỗi lượt dùng train, dừng sớm theo Recall@20 validation, ngày fact của các triple t1 bị xáo trộn
bằng `numpy.random.default_rng(seed)`. Sau đủ 5 lượt, script chấm test 5 seed.

Các lượt hoàn tất được đọc từ checkpoint. Lượt bị ngắt giữa epoch phải chạy lại từ đầu.


In [ ]:
import torch

require_perm_inputs(suite)
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU T4 before Save & Run All.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
if NEW_PERM_FITS_THIS_SESSION < 1:
    raise ValueError('NEW_PERM_FITS_THIS_SESSION must be positive.')

if RUN_TRAINING:
    for step in range(NEW_PERM_FITS_THIS_SESSION):
        if suite['complete']('tkgat_kg_perm'):
            print('tkgat_kg_perm: complete. No other model will be started.', flush=True)
            break
        print(f'Running tkgat_kg_perm only; fit budget {step + 1}/{NEW_PERM_FITS_THIS_SESSION}', flush=True)
        try:
            suite['execute']([
                'scripts/run.py', '--data', suite['DATA_CONFIG'],
                '--model', 'configs/model/tkgat_kg_perm.yaml', '--device', 'cuda',
                '--resume', '--max-new-runs', '1',
            ])
        finally:
            suite['backup'](CHECKPOINT_ZIP)
else:
    print('Training disabled. Recovery verified only.', flush=True)

suite['backup'](CHECKPOINT_ZIP)
print('tkgat_kg_perm:', 'complete' if suite['complete']('tkgat_kg_perm') else 'pending (more fits needed)', flush=True)
print('Save this output:', CHECKPOINT_ZIP, flush=True)


## 3. Lưu kết quả

Tải **`tkgat-lastfm-stage2-perm-results.zip`** từ Output của Version đã hoàn tất.
ZIP có `tkgat_kg_perm_summary.json` (điểm test trung bình ± độ lệch chuẩn mẫu qua 5 seed),
kết quả theo từng user và checkpoint, cùng toàn bộ kết quả Stage 2 đã có trong Recovery.
Nếu log in `pending (more fits needed)`, chạy lại Save & Run All với Recovery là ZIP mới nhất này.
